# Web Crawling for RAG — Building a Crawler from Scratch (Requests + BeautifulSoup + robots.txt)

`l23_web_crawling_for_rag (1).ipynb` compared four ready-made tools (Crawl4AI, Firecrawl, Scrapy,
fastCRW) that each handle link discovery, rendering, and Markdown cleanup for you. This notebook
goes one layer underneath: build the same pipeline — discover pages, fetch them, extract clean
text — using nothing but `requests` and `BeautifulSoup`, the two libraries every one of those
tools is ultimately built on top of.

Two discovery strategies, by what data is available:

| Strategy | Works when | Speed | Completeness |
|---|---|---|---|
| **Sitemap-based** | Site publishes `sitemap.xml` | Fast — one request lists every URL | Only what the sitemap declares |
| **BFS link-following** | Any site, sitemap or not | Slower — one request per page | Finds pages a sitemap might miss, but can wander |

**Target site:** `https://books.toscrape.com` — a public sandbox site built specifically for
scraping practice by the maintainers of Scrapy, so there's no question of permission or load
concerns. It has no `sitemap.xml` (deliberately, to force the BFS path below) and a predictable
catalogue layout, which makes it a good stand-in for "a real site with no sitemap."

**Ground rules followed throughout:**
- Same-domain only (`books.toscrape.com`), starting from the catalogue homepage.
- A small page budget — a dozen pages, not the full 1000-book catalogue.
- `robots.txt` is checked programmatically before any request is sent.
- A polite delay between requests, run sequentially — no concurrency in this notebook (see the
  async extraction-showdown notebook for that).
- A descriptive `User-Agent` identifying this as an educational crawl.


In [ ]:
# pip install requests beautifulsoup4 lxml   (already added via `uv add`)

import time
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

BASE_URL = "https://books.toscrape.com/"
ALLOWED_HOST = "books.toscrape.com"
MAX_PAGES = 12          # small, polite budget
REQUEST_DELAY = 0.5     # seconds between requests
USER_AGENT = "genai-class-educational-crawler/1.0"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

def same_domain(url: str) -> bool:
    return urlparse(url).netloc.lower() == ALLOWED_HOST


## 1. Checking `robots.txt` before crawling

Every tool in the previous notebook either obeyed `robots.txt` internally (Scrapy's
`ROBOTSTXT_OBEY`) or relied on the site explicitly whitelisting crawlers. Doing it by hand uses
Python's own `urllib.robotparser` — a few lines, and it's the same check Scrapy runs under the
hood.


In [ ]:
robots = RobotFileParser()
robots.set_url(urljoin(BASE_URL, "/robots.txt"))
robots.read()

can_fetch = robots.can_fetch(USER_AGENT, BASE_URL)
print(f"Allowed to crawl {BASE_URL}: {can_fetch}")


## 2. Attempting sitemap-based discovery

Checking for a sitemap first is good practice — if it exists, it's a free, complete list of URLs
with no link-following needed at all.


In [ ]:
sitemap_url = urljoin(BASE_URL, "/sitemap.xml")
response = session.get(sitemap_url, timeout=10)

if response.status_code == 200 and "xml" in response.headers.get("Content-Type", ""):
    sitemap_soup = BeautifulSoup(response.text, "xml")
    sitemap_urls = [loc.text for loc in sitemap_soup.find_all("loc")]
    print(f"Sitemap found — {len(sitemap_urls)} URLs listed")
else:
    sitemap_urls = []
    print(
        f"No usable sitemap at {sitemap_url} (status {response.status_code}) "
        "— falling back to BFS link-following"
    )


## 3. BFS link-following crawl

With no sitemap to lean on, the crawler has to discover pages the same way a browser user would:
fetch the homepage, read every `<a href>`, queue up the ones that stay on-domain, and repeat.
This is exactly what Crawl4AI's `BFSDeepCrawlStrategy` and Scrapy's spider both do internally —
here it's a dozen lines of plain Python.


In [ ]:
from collections import deque

def extract_links(html: str, page_url: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    links = []
    for a in soup.find_all("a", href=True):
        absolute = urljoin(page_url, a["href"])
        if same_domain(absolute):
            links.append(absolute.split("#")[0])  # drop in-page anchors
    return links

visited = {}
queue = deque([BASE_URL])
seen = {BASE_URL}

while queue and len(visited) < MAX_PAGES:
    url = queue.popleft()
    try:
        resp = session.get(url, timeout=10)
    except requests.RequestException as exc:
        print(f"  skipped {url}: {exc}")
        continue

    visited[url] = resp.text
    print(f"  fetched ({len(visited)}/{MAX_PAGES}): {url}")

    for link in extract_links(resp.text, url):
        if link not in seen:
            seen.add(link)
            queue.append(link)

    time.sleep(REQUEST_DELAY)

print(f"\n{len(visited)} pages fetched")


## 4. Extraction — raw text vs. targeted CSS selectors

This is the manual equivalent of Crawl4AI's `raw_markdown` vs. `fit_markdown` split. `get_text()`
on the whole `<body>` keeps every nav link, breadcrumb, and sidebar filter the page has — a
targeted CSS selector pulling just the product title/price/description keeps only what actually
matters for RAG.


In [ ]:
sample_url, sample_html = next(iter(visited.items()))
soup = BeautifulSoup(sample_html, "lxml")

raw_text = soup.get_text(separator=" ", strip=True)

title = soup.select_one("div.product_main h1")
price = soup.select_one("p.price_color")
description_tag = soup.select_one("#product_description ~ p")

targeted_text = " | ".join(
    t.get_text(strip=True) for t in [title, price, description_tag] if t
)

print(f"raw get_text():      {len(raw_text):,} chars")
print(f"targeted selectors:  {len(targeted_text):,} chars")
print()
print(targeted_text[:300])


## 5. Building `Document` objects

Same destination as every other crawling notebook in this course: a list of `Document`s with
`page_content` + `metadata`, ready for `l22_chunking.ipynb`.


In [ ]:
documents = []
for url, html in visited.items():
    page_soup = BeautifulSoup(html, "lxml")
    page_title = page_soup.title.get_text(strip=True) if page_soup.title else ""
    body_text = page_soup.get_text(separator=" ", strip=True)
    documents.append(
        Document(
            page_content=body_text,
            metadata={"source": url, "title": page_title, "tool": "requests+bs4"},
        )
    )

print(len(documents), "Documents ready for chunking")
print(documents[0].metadata)


## Summary

| Approach | Setup | Discovery | Extraction quality | When to reach for it |
|---|---|---|---|---|
| **Sitemap-based** | Zero dependencies beyond `requests` | Instant, complete | Depends entirely on what you parse out afterward | Site publishes a sitemap and you want every page, fast |
| **BFS link-following** | Zero dependencies beyond `requests` + `bs4` | One request per page, can miss orphan pages | Same — manual parsing either way | No sitemap, or you only want pages reachable by navigation |
| **Crawl4AI / Firecrawl / Scrapy / fastCRW** *(previous notebook)* | A library or API key | Built-in, handles JS rendering | Markdown cleanup included | Production pipelines, JS-heavy sites, less code to maintain |

## Takeaways

- Every specialized crawling tool is doing the same four things under the hood: check
  `robots.txt`, discover URLs (sitemap or links), fetch pages, extract content. Building it
  manually once makes it obvious what you're trading convenience for.
- `robots.txt` checking takes three lines with `urllib.robotparser` — there's no excuse to skip
  it even in a quick manual crawler.
- The raw-text-vs-targeted-selector gap here is the same gap as `raw_markdown` vs. `fit_markdown`
  in Crawl4AI — less automated, but it shows exactly where the "noise" a RAG pipeline doesn't
  want comes from.
- A from-scratch crawler is reasonable for a handful of pages on a simple, static site; past
  that, the JS-rendering, retry/backoff, and anti-bot handling that Crawl4AI/Firecrawl/Scrapy
  provide stop being optional.
